# Word2Vec

## Setup and Imports

In [25]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.io as pio

from gensim.corpora import Dictionary
from gensim.models import word2vec

from sklearn.manifold import TSNE as tsne

In [26]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [27]:
import gensim
gensim.__version__

'4.3.3'

In [28]:
OHCO = ['doc_title','para_num','sentence_num','token_num']
BAG = OHCO[:3]


In [29]:
TOKENS = pd.read_csv('data/p2591-TOKENS.csv').set_index(OHCO).dropna() # Dropped nan tokens because it was causing errors in embeding
TOKENS.head()

pos_tuple pos token_str  \
doc_title para_num sentence_num token_num                                 
ASHPUTTEL 0        0            0           ('The', 'DT')  DT       The   
                                1          ('wife', 'NN')  NN      wife   
                                2            ('of', 'IN')  IN        of   
                                3             ('a', 'DT')  DT         a   
                                4          ('rich', 'JJ')  JJ      rich   

                                          term_str pos_group  
doc_title para_num sentence_num token_num                     
ASHPUTTEL 0        0            0              the        DT  
                                1             wife        NN  
                                2               of        IN  
                                3                a        DT  
                                4             rich        JJ

## Convert to Gensim

In [30]:
docs = TOKENS.groupby(BAG).term_str.apply(list).tolist()

In [31]:
for i in range(5):
    print(f"Doc {i}:", docs[i])

Doc 0: ['the', 'wife', 'of', 'a', 'rich', 'man', 'fell', 'sick']
Doc 1: ['and', 'when', 'she', 'felt', 'that', 'her', 'end', 'drew', 'nigh', 'she', 'called', 'her', 'only', 'daughter', 'to', 'her', 'bedside', 'and', 'said', 'always', 'be', 'a', 'good', 'girl', 'and', 'i', 'will', 'look', 'down', 'from', 'heaven', 'and', 'watch', 'over', 'you']
Doc 2: ['soon', 'afterwards', 'she', 'shut', 'her', 'eyes', 'and', 'died', 'and', 'was', 'buried', 'in', 'the', 'garden']
Doc 3: ['and', 'the', 'little', 'girl', 'went', 'every', 'day', 'to', 'her', 'grave', 'and', 'wept', 'and', 'was', 'always', 'good', 'and', 'kind', 'to', 'all', 'about', 'her']
Doc 4: ['and', 'the', 'snow', 'fell', 'and', 'spread', 'a', 'beautiful', 'white', 'covering', 'over', 'the', 'grave']


In [32]:
dictionary = Dictionary(docs) 


## Generate Embeddings

In [33]:
w2v_params = dict(
    window = 2,
    vector_size = 200,
    min_count = 50, 
    workers = 4
)

In [34]:
model = word2vec.Word2Vec(docs, **w2v_params)
model.wv.vectors

array([[-0.14506595,  0.14224172, -0.06113003, ..., -0.16331632,
        -0.10850887, -0.14606199],
       [-0.11093624,  0.10349076, -0.07917537, ..., -0.18228304,
        -0.0726064 , -0.13925408],
       [-0.01241606, -0.00513959, -0.07419924, ..., -0.2203813 ,
         0.00096863, -0.09242472],
       ...,
       [-0.01673332,  0.00887487, -0.06155762, ..., -0.19566059,
        -0.01364566, -0.08084812],
       [-0.01417441,  0.00441522, -0.06363907, ..., -0.19425786,
        -0.02354462, -0.0805377 ],
       [ 0.0249457 , -0.04135813, -0.06470067, ..., -0.21216388,
         0.00052452, -0.05982339]], dtype=float32)

In [35]:
WV = pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key)
WV.index.name = 'term_str'
WV

,0,1,2,3,4,5,6,7,8,9,...,190,191,192,193,194,195,196,197,198,199
term_str,,,,,,,,,,,,,,,,,,,,,
the,-0.145066,0.142242,-0.061130,0.035077,0.029211,0.034716,0.199557,0.293350,-0.057304,0.189440,...,0.090594,0.044621,-0.103981,-0.211532,0.137296,-0.009110,0.100827,-0.163316,-0.108509,-0.146062
and,-0.110936,0.103491,-0.079175,0.055773,0.069892,0.008225,0.154321,0.269810,-0.068363,0.170579,...,0.076303,0.047759,-0.113310,-0.201108,0.148550,-0.000144,0.097865,-0.182283,-0.072606,-0.139254
to,-0.012416,-0.005140,-0.074199,0.141933,0.130296,-0.082950,0.099918,0.238728,-0.078695,0.116425,...,0.077735,0.040325,-0.095313,-0.153953,0.122329,0.005639,0.108051,-0.220381,0.000969,-0.092425
he,-0.041265,0.045786,-0.090604,0.088498,0.100984,-0.031359,0.151757,0.236133,-0.047027,0.138723,...,0.105896,0.022295,-0.110631,-0.163461,0.164765,0.031477,0.102400,-0.238126,-0.028538,-0.136429
a,-0.039378,0.032157,-0.048872,0.149817,0.131615,-0.060556,0.157195,0.257790,-0.099215,0.128089,...,0.068553,0.024727,-0.083985,-0.126191,0.113923,-0.018941,0.086737,-0.186217,-0.021255,-0.090421
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
wished,-0.005345,-0.011825,-0.065072,0.149824,0.128816,-0.078715,0.101242,0.228933,-0.066017,0.100485,...,0.058849,0.020931,-0.072421,-0.124339,0.099873,0.006672,0.092607,-0.189977,-0.013543,-0.081156
better,0.028577,-0.040368,-0.066803,0.192960,0.173300,-0.121288,0.103467,0.244353,-0.084463,0.085899,...,0.063158,0.007248,-0.073087,-0.114938,0.099542,0.013894,0.108847,-0.227697,0.004043,-0.070622
third,-0.016733,0.008875,-0.061558,0.131006,0.119066,-0.067153,0.110771,0.229864,-0.066186,0.107248,...,0.068197,0.013203,-0.076255,-0.130117,0.112384,0.008742,0.087356,-0.195661,-0.013646,-0.080848


## Make and Plot TSNE

In [44]:
PP = 50 #40 # Try 1, 100, etc.

tsne_engine = tsne(
    perplexity=PP, 
    n_components=2, 
    init='pca', 
    max_iter=2500, 
    random_state=23
)
TSNE = pd.DataFrame(
    tsne_engine.fit_transform(WV), 
    columns=['x','y'], 
    index=WV.index)
TSNE

,x,y
term_str,,
the,-13.328802,3.399593
and,-12.854936,3.195250
to,2.793377,-2.276005
he,-8.937138,4.892931
a,-5.579622,-2.038567
...,...,...
wished,3.290664,1.391510
better,9.539525,-0.320192
third,-0.326009,1.927955


In [45]:
px.scatter(TSNE.reset_index(), 'x', 'y', 
        text='term_str', 
        hover_name='term_str',  
        # size='n',
        height=1000,
        width=1200)\
    .update_traces(
        mode='markers+text', 
        textfont=dict(color='black', size=14, family='Arial'),
        textposition='top center')

## Save Outputs

In [46]:
WV.to_csv('output/p2591-WORD2VEC.csv')